# Online DSpark Training for Llama-3.2-1B-Instruct

This notebook trains a [DSpark](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/algorithms/dspark/)
speculator (the *draft* model) for `unsloth/Llama-3.2-1B-Instruct` (the *verifier*, or target).
The drafter is a 2-layer, ~0.25B-parameter model that proposes a block of 4 tokens at a time; the
verifier checks the whole block in a single forward pass and keeps the longest correct prefix. Since
every kept token is one the verifier itself would have produced, decoding is faster but the output
distribution is unchanged.

`speculators` implements several draft architectures (Eagle-3, P-EAGLE, DFlash, DFlash2, DSpark,
MTP). **DSpark** builds on DFlash's block-based diffusion decoder and adds a Markov logit bias plus
per-position confidence heads, which is what lets it propose a whole block of `--block-size` tokens
at once rather than one token at a time.

**Online training.** A drafter conditions on the verifier's *hidden states*, which have to come from
somewhere. `speculators` offers three modes:

| Mode | Hidden states | Trade-off |
| --- | --- | --- |
| offline | all pre-generated to disk before training | fastest epochs, enormous disk footprint |
| hybrid | generated during epoch 0, cached and reused | middle ground |
| **online** (used here) | fetched from a live vLLM server per batch, then discarded | near-zero disk, needs the verifier resident throughout |

Online is the right pick on Kaggle, where disk is small and sessions are ephemeral. The cost is that
two GPUs are needed at once: the verifier serves hidden states on GPU 0 while training runs on GPU 1.

**Inputs and outputs.** The training data is the on-policy JSONL produced by
[`data-preparation.ipynb`](./data-preparation.ipynb) and published to
[`yosefw/magpie-llama-3.2-1b-instruct`](https://huggingface.co/datasets/yosefw/magpie-llama-3.2-1b-instruct).
The output is a self-contained drafter checkpoint pushed to the Hub and benchmarked with vLLM at the
end of this notebook.

**Environment.** Written for a Kaggle notebook with 2 GPUs (e.g. T4 x2), internet enabled, and a
Hugging Face token with write scope stored in Kaggle Secrets. Budget a long session - the published
model took roughly 10-11 hours end to end.

**Reference.**
[Speculators training tutorial](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train/)

## Step 0 - Set up the environment

Everything in this notebook is driven by scripts that live in the `speculators` repository under
`scripts/` (`prepare_data.py`, `launch_vllm.py`, `train.py`, `evaluate/`). Those are not shipped in
the published wheel, so the repository is cloned rather than `pip install speculators`. The `%cd`
below is persistent for the rest of the notebook: every path from here on is relative to the
checkout.

In [ ]:
!git clone https://github.com/vllm-project/speculators.git

In [ ]:
%cd speculators

### Install vLLM and speculators

| Command | Why |
| --- | --- |
| `uv venv speculators_venv` | Creates the venv the tutorial names, kept for parity with the docs. |
| `uv pip install "vllm>=0.22.0"` | Serves the verifier, supplies hidden states, and later serves the trained drafter. |
| `uv pip install -e .` | Installs `speculators` from the clone, so the in-tree scripts are importable. |

The upstream tutorial keeps training and serving in two separate venvs to stop the training stack
from fighting vLLM over torch versions. Here they share one, which is what online training needs
anyway - `train.py` talks to the server over HTTP, so only one torch install is ever active.

Kaggle caveat: each `!` line runs in its own shell, so `source speculators_venv/bin/activate` does
not carry over to the next line or cell. The packages land in the notebook's own Python environment,
which is what the later cells actually run against. The venv is therefore nominal.

In [ ]:
# Speculators venv (for data prep and training)
! uv venv speculators_venv
! source speculators_venv/bin/activate
! uv pip install "vllm>=0.22.0"
! uv pip install -e .

### Fix up the preinstalled Kaggle packages

Kaggle images ship a `torchaudio` build pinned to their own torch version, which conflicts with the
torch vLLM pulls in and breaks the import chain; nothing here needs audio, so it is removed.
`datasets` is then upgraded because the loaders used downstream need a newer version than the image
provides - `--break-system-packages` is required to overwrite the distribution-managed copy.

In [ ]:
! pip uninstall -y torchaudio
! pip install -Uq datasets --break-system-packages

## Step 1 - Fetch the on-policy responses

Training data for a speculator has to be **on-policy**: generated by the exact verifier the drafter
will be paired with, so the drafter learns to predict what that model would actually say.
`data-preparation.ipynb` did that generation once and pushed the result to the Hub, which makes it a
stable, versioned input - retraining never re-runs generation, and a fresh Kaggle session just
downloads the file again.

First, authentication. The token is read from Kaggle Secrets into `HF_TOKEN`, where
`huggingface_hub` picks it up automatically. The secret must be attached to this notebook
(Add-ons -> Secrets) and needs **write** scope, since the same token is reused in Step 5 to create a
model repo. Off Kaggle the `kaggle_secrets` import fails; replace this cell with a direct
`os.environ["HF_TOKEN"] = "hf_..."`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_WRITE_YOSEFW")

### Download the JSONL

`local_dir="."` drops the file in the repository root instead of the Hub cache, so the path handed
to `prepare_data.py` in the next step is a plain relative filename. Each row of the JSONL is one
conversation already in speculator format: `input_ids` plus a `loss_mask` marking the positions the
drafter is scored on.

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="yosefw/magpie-llama-3.2-1b-instruct",
    filename="magpie_Llama-3.2-1B-Instruct.jsonl",
    repo_type="dataset",
    local_dir=".",          # drop it in the cwd instead of the cache
)
print(path)

## Step 2 - Tokenize into an Arrow dataset

`prepare_data.py` converts the JSONL into preprocessed Arrow files - fixed-length token sequences
with their loss masks - which is the format the training loop's dataloader reads. This is Step 1 of
the upstream tutorial.

| Flag | Meaning |
| --- | --- |
| `--model` | The verifier, used here for its tokenizer and chat template. |
| `--data` | The JSONL downloaded in Step 1. |
| `--output ./output` | Destination for the Arrow shards; `train.py` reads this same directory. |
| `--max-samples 32000` | Cap on conversations processed. Drop to a few hundred to smoke-test the pipeline. |
| `--seq-length 2048` | Maximum sequence length; longer conversations are truncated. Must match `--total-seq-len` in Step 4. |
| `--overwrite` | Replaces an existing `./output`, so the cell is safe to re-run. |

The commented-out `rm -r ./output` above is a manual reset kept for convenience; `--overwrite` makes
it unnecessary.

In [ ]:
# ! rm -r ./output

In [ ]:
# in speculators venv
! python scripts/prepare_data.py \
  --model "unsloth/Llama-3.2-1B-Instruct" \
  --data ./magpie_Llama-3.2-1B-Instruct.jsonl \
  --output ./output \
  --max-samples 32000 \
  --seq-length 2048 \
  --overwrite

## Step 3 - Serve the verifier for hidden states

`launch_vllm.py` is a thin wrapper around `vllm serve` that turns on hidden-state export, so the
server returns not just tokens but the intermediate activations the drafter is trained to condition
on. It stays up for the whole of Step 4.

`--target-layer-ids 2 5 8 11 14` selects which transformer layers are exported. **This list must be
identical to the one passed to `train.py` in Step 4.** There is no cross-check between the two - a
mismatch means the drafter asks for layers the server is not exporting, and the run either errors
out mid-epoch or quietly learns from the wrong signal.

Two details in the launch code:

- `start_new_session=True` puts the server in its own process group, so interrupting a notebook cell
  (SIGINT) does not take the server down with it. It is stopped explicitly after training instead.
- The log path `/content/vllm.log` is a Colab convention. On Kaggle `/content` may not exist - point
  this at something under `/kaggle/working/` if the cell fails to open the file.

In [ ]:
import subprocess, json

log_file = open("/content/vllm.log", "w")
process = subprocess.Popen(
    [
        "python", "scripts/launch_vllm.py", 
        "unsloth/Llama-3.2-1B-Instruct",
        "--target-layer-ids", "2", "5", "8", "11", "14",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,  # detaches from the notebook's process group so a cell interrupt (SIGINT) doesn't reach it
)
print(f"Started vLLM server with PID {process.pid}")

### Wait until the server is ready

Loading the verifier and allocating the KV cache takes minutes. Training would fail immediately
against a server that is not yet listening, so this cell polls `/health` every 5 seconds for up to
5 minutes and only returns once vLLM reports ready. If it exhausts the loop without printing
`Server is ready!`, read the log file named above - an OOM or a bad model id will be at the bottom
of it.

In [ ]:
import time, requests

for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

## Step 4 - Train the DSpark drafter

The main event. `CUDA_VISIBLE_DEVICES=1` pins training to the second GPU while vLLM keeps the first;
`torchrun --standalone --nproc_per_node 1` runs a single training process (raise `--nproc_per_node`
if more training GPUs are available).

| Flag | Meaning |
| --- | --- |
| `--verifier-name-or-path` | The model being accelerated. Its config determines the drafter's hidden size and embedding table. |
| `--data-path ./output` | The Arrow dataset from Step 2. |
| `--save-path ./output/checkpoints` | Where per-epoch checkpoints are written. |
| `--draft-vocab-size 32000` | The drafter predicts over a reduced 32K vocabulary instead of the verifier's 128K, which shrinks its output head substantially at a small cost in coverage. |
| `--epochs 6` | Passes over the training split. |
| `--train-data-ratio 0.96` | 96/4 train/validation split; validation loss is what picks `checkpoint_best`. |
| `--total-seq-len 2048` | Training sequence length. Match `--seq-length` from Step 2. |
| `--max-anchors 128` | Cap on the positions per sequence the drafter is trained from. Lower values cut memory and step time. |
| `--num-workers 4` | Dataloader workers. |
| `--speculator-type dspark` | Selects the DSpark algorithm (alternatives include `eagle3`, `peagle`, `dflash`, `mtp`). |
| `--num-layers 2` | Depth of the drafter. |
| `--block-size 4` | Tokens proposed per verification round - the drafter's speculation length. |
| `--optimizer adamw` / `--lr 6e-4` | Optimizer and peak learning rate. |
| `--loss-fn '{"ce": 0.1, "tv": 0.9}'` | DSpark's loss mixture: cross-entropy for token prediction plus a total-variation term for confidence calibration, weighted here heavily toward TV. |
| `--vllm-endpoint` | The server from Step 3. Its presence is what makes this an online run. |
| `--target-layer-ids 2 5 8 11 14` | Layers to read hidden states from. **Must match Step 3.** |
| `--on-missing generate` | When a batch's hidden states are absent, request them from the server rather than raising or skipping. |
| `--on-generate delete` | Discard them after the step instead of caching to disk. Together with the flag above, this pair is what defines online mode. |
| `--no-resume-from-checkpoint` | Start fresh rather than picking up an existing checkpoint in `--save-path`. |
| `--log-freq 200` | Steps between log lines. |

Expect hours of runtime - roughly 10 h for the published model on Kaggle's 2x T4. Output is one
directory per epoch under `./output/checkpoints`, each with `config.json`, `model.safetensors`, and
optimizer/scheduler state, plus a `checkpoint_best` symlink pointing at the lowest validation loss.

In [ ]:
# in speculators venv
! CUDA_VISIBLE_DEVICES=1 torchrun --standalone --nproc_per_node 1 \
  scripts/train.py \
  --verifier-name-or-path "unsloth/Llama-3.2-1B-Instruct" \
  --data-path ./output \
  --save-path ./output/checkpoints \
  --draft-vocab-size 32000 \
  --epochs 6 \
  --train-data-ratio 0.96 \
  --total-seq-len 2048 \
  --max-anchors 128 \
  --num-workers 4 \
  --speculator-type dspark \
  --num-layers 2 \
  --block-size 4 \
  --optimizer adamw \
  --lr 6e-4 \
  --loss-fn '{"ce": 0.1, "tv": 0.9}' \
  --vllm-endpoint http://localhost:8000/v1 \
  --target-layer-ids 2 5 8 11 14 \
  --on-missing generate \
  --on-generate delete \
  --no-resume-from-checkpoint \
  --log-freq 200

### Shut down the verifier

`killpg` rather than `kill`, because `start_new_session=True` in Step 3 put the server in its own
process group - signalling only the parent would leave vLLM's worker processes running and holding
GPU memory. That memory has to be released before the drafter can be served in Step 6.

In [ ]:
import os, signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)

## Step 5 - Publish the checkpoint

The Kaggle secret is read again here because training may have run in an earlier session, or the
kernel may have restarted since Step 1; the cell is a no-op if `HF_TOKEN` is already set.

`checkpoint_best` is self-contained - it carries the drafter weights plus a config recording which
verifier it belongs to - so uploading that directory as-is produces a repository vLLM can serve
directly. `create_repo(..., exist_ok=True)` makes re-runs idempotent.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_WRITE_YOSEFW")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id="yosefw/Llama-3.2-1B-Instruct-DSpark", exist_ok=True)

api.upload_folder(
    folder_path="/kaggle/working/speculators/output/checkpoints/checkpoint_best",
    repo_id="yosefw/Llama-3.2-1B-Instruct-DSpark",
    repo_type="model",
)

## Step 6 - Serve and benchmark

The evaluation harness lives in `scripts/evaluate` with its own requirements. Note the `%cd` - the
remaining cells run from inside that directory.

In [ ]:
%cd scripts/evaluate
! pip install -r requirements.txt

### Serve the trained drafter

Serving the *drafter* repository is enough: its config names the verifier, which vLLM downloads and
loads automatically, then wires the two together for speculative decoding. The drafter is not a
standalone model and will not generate anything on its own. The checkpoint also carries a
`speculators_config` block, so vLLM turns speculative decoding on by itself - no
`--speculative-config` argument is required.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` and `--gpu-memory-utilization 0.80` guard against
fragmentation OOM now that two sets of weights share one card. As in Step 3, the server is detached
with `start_new_session=True` and the cell blocks on a `/health` poll until it is ready.

In [ ]:
import subprocess, os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log_file = open("vllm_serve.log", "w")
process = subprocess.Popen(
    [
        "vllm", "serve", "yosefw/Llama-3.2-1B-Instruct-DSpark",
        "--port", "8000",
        "--gpu-memory-utilization", "0.80"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,  # detaches from notebook's process group so cell interrupts don't kill it
)
print(f"Started vLLM server with PID {process.pid}")

import time, requests

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

### Measure acceptance

`evaluate.py throughput` replays a fixed benchmark suite through the endpoint and reports, per
subset:

- **`acceptance_length`** - the mean number of tokens kept per verification round. With
  `--block-size 4` the ceiling is 5 (the proposed block plus the verifier's own token) and 1.0 means
  nothing was ever accepted. This is an *upper bound* on single-stream speedup: it does not charge
  for the drafter's own forward pass.
- **`pos_0` .. `pos_3`** - the acceptance rate at each slot of the proposed block. These decay
  across the block by construction, since a slot can only be accepted if every slot before it was.
  They use a different denominator from `acceptance_length`, so they do not sum to it.

Acceptance is highest on structured, predictable text - code, math, tool calls - and lowest on
translation and summarization.

In [ ]:
! python evaluate.py throughput --target http://localhost:8000/v1

## Results and next steps

For the published run, weighted over 117,372 verification steps, acceptance length came out at
**2.404**, ranging from **2.927** on HumanEval down to **1.736** on summarization. The full
per-subset table is in the [README](../README.md); wall-clock throughput was not measured.

Serve the finished drafter anywhere vLLM runs:

```bash
vllm serve yosefw/Llama-3.2-1B-Instruct-DSpark \
  --port 8000 \
  --gpu-memory-utilization 0.75
```

Then query the OpenAI-compatible endpoint at `http://localhost:8000/v1` as usual - speculative
decoding is transparent to the caller.

To push acceptance higher, the highest-leverage knobs are more on-policy data (regenerate with a
larger `--limit` in `data-preparation.ipynb`), a longer sequence length, and more epochs.